# **EXERCISE 1**

In [ ]:
!pip install gensim

In [ ]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/NLP2025/yelp/yelp/train_en.txt', sep='\t')

In [ ]:
# df.sample(10)
df.head(10)

In [ ]:
df['Tokens'] = df['Sentence'].apply(lambda x: word_tokenize(str(x)))
df['Tokens'].head(10)
sentences = df['Tokens'].values.tolist()
sentences

In [ ]:
df['Tokens'].head(10)

**Initializing and Training a Skip-Gram Word2Vec Model with 100-Dimensional Vectors**

In [ ]:
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=15,
    sg=1,
    workers=4,
    seed=42
)

In [ ]:
len(model.wv.index_to_key)

In [ ]:
len(model.wv.vectors)

In [ ]:
model.wv.index_to_key

In [ ]:
model.wv.vectors

**Visualizing Word Embeddings in 2D: PCA Projection of Word2Vec Vectors**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

words = list(model.wv.index_to_key)
vectors = model.wv[words]

pca = PCA(n_components=2)
vecs_2d = pca.fit_transform(vectors)

plt.figure(figsize=(12, 8))
plt.scatter(vecs_2d[:, 0], vecs_2d[:, 1], s=10, alpha=0.6)


for i, word in enumerate(words[:60]):
    plt.annotate(word, xy=(vecs_2d[i, 0], vecs_2d[i, 1]), fontsize=9, alpha=0.7)

plt.title("Визуелизација на Word2Vec зборовни вектори (2D)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()


**2D Visualization of Word2Vec Word Vectors using t-SNE (Top 60 Words)**

In [ ]:
tsne = TSNE(n_components=2, perplexity=40, random_state=42)
vecs_2d = tsne.fit_transform(vectors[:60])
plt.figure(figsize=(12, 8))
plt.scatter(vecs_2d[:, 0], vecs_2d[:, 1], s=10, alpha=0.6)


for i, word in enumerate(words[:60]):
    plt.annotate(word, xy=(vecs_2d[i, 0], vecs_2d[i, 1]), fontsize=9, alpha=0.7)

plt.title("Визуелизација на Word2Vec зборовни вектори (2D)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()

**Function to Perform Word Analogy Operations on Word2Vec Embeddings**

In [ ]:
def vector_arithmetic(model, positive=[], negative=[]):

    positive = [w for w in positive if w in model.wv]
    negative = [w for w in negative if w in model.wv]

    if not positive:
        return "No positive words in vocabulary!"

    try:
        result = model.wv.most_similar(positive=positive, negative=negative, topn=1)
        return result[0][0]
    except KeyError as e:
        return f"Word not in vocabulary: {e}"

In [ ]:
operations = [
    (["Paris", "Italy"], ["France"]),
    (["Madrid", "France"], ["Spain"]),
    (["King", "Woman"], ["Man"]),
    (["Bigger", "Cold"], ["Big"]),
    (["Windows", "Google"], ["Microsoft"])
]

**Initializing and Training a CBOW Word2Vec Model with 100-Dimensional Vectors**

In [ ]:
model2 = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=15,
    sg=0,
    workers=4,
    seed=42
)

**Performing Word Analogy Operations on the Skip-Gram Word2Vec Model**

In [ ]:
for pos, neg in operations:
    output = vector_arithmetic(model, positive=pos, negative=neg)
    print(f"{' + '.join(pos)} - {' + '.join(neg)} = {output}")

**Performing Word Analogy Operations on the CBOW Word2Vec Model**

In [ ]:
for pos, neg in operations:
    output = vector_arithmetic(model2, positive=pos, negative=neg)
    print(f"{' + '.join(pos)} - {' + '.join(neg)} = {output}")

**Function to Train Skip-Gram and CBOW Word2Vec Models Simultaneously with Specified Vector Size and Parameters**

In [ ]:
def train_two_models(sentences, vector_size=120, window=5, min_count=15, workers=4):
    skip_gram_model = Word2Vec(
        sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,
        workers=workers
    )

    cbow_model = Word2Vec(
        sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=0,
        workers=workers
    )

    return {'skip_gram': skip_gram_model, 'cbow': cbow_model}

In [ ]:
models_120 = train_two_models(sentences)

**Performing Word Analogy Operations on the 120-Dimensional Skip-Gram Model**

In [ ]:
for pos, neg in operations:
    output = vector_arithmetic(models_120['skip_gram'], positive=pos, negative=neg)
    print(f"{' + '.join(pos)} - {' + '.join(neg)} = {output}")

**Performing Word Analogy Operations on the 120-Dimensional CBOW Model**

In [ ]:
for pos, neg in operations:
    output = vector_arithmetic(models_120['cbow'], positive=pos, negative=neg)
    print(f"{' + '.join(pos)} - {' + '.join(neg)} = {output}")

# **EXERCISE 2**

In [ ]:
df.head(10)

**Creating a Vocabulary with Minimum Word Frequency and <UNK> Token**

In [ ]:
def create_vocabulary_from_w2v(model):

    vocab = model.wv.index_to_key.copy()
    vocab.append('<UNK>')

    w_to_i = {word: idx for idx, word in enumerate(vocab)}
    i_to_w = {idx: word for idx, word in enumerate(vocab)}

    return vocab, w_to_i, i_to_w


In [ ]:
vocab, w_to_i, i_to_w = create_vocabulary_from_w2v(model)

print("Vocabulary:", vocab)
print("Word to Index:", w_to_i)
print("Index to Word:", i_to_w)

In [ ]:
unk_idx = w_to_i['<UNK>']
df['Tokens_ID'] = df['Tokens'].apply(lambda x: [w_to_i[word] if word in w_to_i else unk_idx for word in x])
df

**Identifying Samples Containing `<UNK>` Tokens in Tokenized Sequences**

In [ ]:
unk_rows = df[df['Tokens_ID'].apply(lambda x: unk_idx in x)]
unk_rows['Tokens_ID']

**Determine the Average Token Sequence Length to Use for pad_sequences**

In [ ]:
avg_len = int(np.round(df['Tokens_ID'].apply(len).mean()))
print("Average sequence length:", avg_len)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy
from sklearn.model_selection import train_test_split

**Append Randomly Initialized Vector for Unknown `(<UNK>)` Token to Word2Vec Embeddings**



In [ ]:
word_vectors = model.wv.vectors
words = model.wv.index_to_key

unk_vector = np.random.normal(scale=0.6, size=(word_vectors.shape[1],))
word_vectors = np.vstack([word_vectors, unk_vector])

**Preprocess Validation Set:**

In [ ]:
df_val = pd.read_csv('/content/drive/MyDrive/NLP2025/yelp/yelp/val_en.txt', sep='\t')
df_val.head(10)

In [ ]:
df_val['Tokens'] = df_val['Sentence'].apply(lambda x: word_tokenize(str(x)))
df_val['Tokens'].head(10)

In [ ]:
unk_idx = w_to_i['<UNK>']
df_val['Tokens_ID'] = df_val['Tokens'].apply(lambda x: [w_to_i[word] if word in w_to_i else unk_idx for word in x])
df_val

**Pad Training Sequences and Prepare Labels**

In [ ]:
from keras.preprocessing.sequence import pad_sequences
import numpy as np
X_train = pad_sequences(df['Tokens_ID'], maxlen=avg_len, padding='post', truncating='post')
Y_train = np.array([1 if label=='positive' else 0 for label in df['Style']])

**Pad Validation Sequences and Prepare Labels**

In [ ]:
X_val = pad_sequences(df_val['Tokens_ID'], maxlen=avg_len, padding='post', truncating='post')
Y_val= np.array([1 if label=='positive' else 0 for label in df_val['Style']])

**Define and Compile LSTM Sentiment Analysis Model with Pretrained Word2Vec Embeddings**

In [ ]:
rnn_model = Sequential()
rnn_model.add(Embedding(
        input_dim=word_vectors.shape[0],
        output_dim=word_vectors.shape[1],
        weights=[word_vectors],
        trainable=True
    ))
rnn_model.add(LSTM(128))
rnn_model.add(Dense(1, activation='sigmoid'))

rnn_model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

**Train LSTM Model on Training Data with Validation Monitoring**

In [ ]:
history = rnn_model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)

**Training and Validation Performance Visualization**

In [ ]:
import matplotlib.pyplot as plt

# plt.plot(history.history['loss'], label='train_loss')
# plt.plot(history.history['val_loss'], label='val_loss')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()
# plt.show()

plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


**Preprocessing Test Data**

In [ ]:
df_test = pd.read_csv(
    '/content/drive/MyDrive/NLP2025/yelp/yelp/test_en.txt',
    sep='\t',
    names=['Sentence','Style','User'],
    header=0,
    on_bad_lines='skip',
    engine='python'
)
df_test['Tokens'] = df_test['Sentence'].apply(lambda x: word_tokenize(str(x)))
df_test['Tokens_ID'] = df_test['Tokens'].apply(lambda x: [w_to_i[word] if word in w_to_i else unk_idx for word in x])
df_test

**EVALUATION**

In [ ]:
X_test = pad_sequences(df_test['Tokens_ID'], maxlen=avg_len, padding='post', truncating='post')
Y_test = np.array([1 if label=='positive' else 0 for label in df_test['Style']])

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = (rnn_model.predict(X_test) > 0.5).astype(int)

print("Accuracy:", accuracy_score(Y_test, y_pred))
print("Precision:", precision_score(Y_test, y_pred))
print("Recall:", recall_score(Y_test, y_pred))
print("F1 score:", f1_score(Y_test, y_pred))